In [25]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [26]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [27]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, sleep_input_size, num_layers=1, num_layers_sleep=1):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(sleep_input_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        # self.fc = nn.Linear(hidden_wake_size, 15)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        # out_ = self.fc(out) 
        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw


In [28]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [29]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [30]:
class place_cells(nn.Module):
    def __init__(self, input_size, output_size):
        super(place_cells, self).__init__()
        
        self.linear1 = nn.utils.weight_norm(nn.Linear(input_size, output_size))

    def forward(self, x):
        out = self.linear1(x)

        return out

In [32]:
### initial training ###
total_samples = 40000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 10
sleep_output_size = 5
sleep_input_size=  20
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, sleep_input_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 2.0665, accuracy: 0.2350
Iter : 2001, loss: 2.0236, accuracy: 0.2500
Iter : 3001, loss: 1.8534, accuracy: 0.3020
Iter : 4001, loss: 2.3737, accuracy: 0.5270
Iter : 5001, loss: 1.7975, accuracy: 0.6080
Iter : 6001, loss: 2.0696, accuracy: 0.6510
Iter : 7001, loss: 2.6600, accuracy: 0.6630
Iter : 8001, loss: 1.5491, accuracy: 0.6450
Iter : 9001, loss: 1.5183, accuracy: 0.6540
Iter : 10001, loss: 1.2158, accuracy: 0.6650
Iter : 11001, loss: 1.8352, accuracy: 0.6750
Iter : 12001, loss: 1.4982, accuracy: 0.6660
Iter : 13001, loss: 2.3703, accuracy: 0.6660
Iter : 14001, loss: 1.7011, accuracy: 0.6690
Iter : 15001, loss: 2.4353, accuracy: 0.6650
Iter : 16001, loss: 1.6454, accuracy: 0.6670
Iter : 17001, loss: 1.3607, accuracy: 0.6670
Iter : 18001, loss: 1.8634, accuracy: 0.6800
Iter : 19001, loss: 1.6475, accuracy: 0.6580
Iter : 20001, loss: 2.3107, accuracy: 0.6720
Iter : 21001, loss: 1.8899, accuracy: 0.6740
Iter : 22001, loss: 1.7706, accuracy: 0.6570
Iter : 23001, loss:

In [71]:
def mse_contrastive_loss(pivot, pos, neg, repel_margin=50.0):
    pull_loss = nn.functional.mse_loss(pivot, pos)
    d_neg = ((pivot.unsqueeze(1) - neg) ** 2).sum(dim=-1) 
    repel_loss = torch.relu(repel_margin - d_neg).mean()

    return pull_loss + repel_loss


In [75]:
total_samples = 200000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
counts = []
seq = ''
seq_ = ''

place_cell = place_cells(hidden_wake_size, sleep_output_size )
optimizer = torch.optim.SGD(place_cell.parameters(), lr=1e-3, momentum=0.95)
criterion = mse_contrastive_loss

for jj in range(total_samples):
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1))
        else:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hidden_state)

        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()

        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]


        #### Mine positive and negative examples ####
        X_pos = torch.zeros(len(tokens),dtype=torch.float32)
        idx = X_hat_prob.argmax()
        X_pos[idx] = 1.0
        X_pos = X_pos.reshape(1,1,-1)
        _, hidden_state_pos = network1(X_pos, hidden_state)

        X_neg = torch.zeros(len(tokens),dtype=torch.float32)
        idx = X_hat_prob.argmin()
        X_neg[idx] = 1.0
        X_neg = X_neg.reshape(1,1,-1)
        _, hidden_state_neg = network1(X_neg, hidden_state)
 
        
    optimizer.zero_grad()

    out1 = place_cell(hidden_state.reshape(-1))
    out2 = place_cell(hidden_state_pos.reshape(-1))
    out_neg = place_cell(hidden_state_neg.reshape(-1))
    
    loss = criterion(out1,out2,out_neg)
    loss.backward(retain_graph=True)
    optimizer.step()

    if jj%10000==0:
        print('Loss',loss.item())
    
        

Loss 47.348873138427734


In [73]:
total_samples = 20
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
counts = []
x_sleep = torch.zeros(sleep_inputput_size,dtype=torch.float32)
seq = ''

for jj in range(total_samples):
    with torch.no_grad():
        # print(X_hat)
        print(tokens[X_hat.argmax()])
        if jj == 0:
            X_hat_, hidden_state, hs = network1(X_hat.reshape(1,1,-1), x_=x_sleep.reshape(1,1,-1), sleep=True)
        else:
            X_hat_, hidden_state, hs = network1(X_hat.reshape(1,1,-1), x_=x_sleep.reshape(1,1,-1), hw=hidden_state, hs=hs, sleep=True)

        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        # print(X_hat_prob)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()


        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]


        #### Mine positive and negative examples ####
        X_pos = torch.zeros(len(tokens),dtype=torch.float32)
        idx = X_hat_prob.argmax()
        X_pos[idx] = 1.0
        X_pos = X_pos.reshape(1,1,-1)
        _, hidden_state_pos = network1(X_pos, hidden_state)
        print(tokens[idx], ' max', idx)

        X_neg = torch.zeros(len(tokens),dtype=torch.float32)
        idx = X_hat_prob.argmin()
        X_neg[idx] = 1.0
        X_neg = X_neg.reshape(1,1,-1)
        _, hidden_state_neg = network1(X_neg, hidden_state)
        print(tokens[idx], ' min', idx)

        out1 = place_cell(hidden_state.reshape(-1))
        x_sleep = out1
        out2 = place_cell(hidden_state_pos.reshape(-1))
        out_neg = place_cell(hidden_state_neg.reshape(-1))
        
        print(out1, out2, out_neg)
    
        

A
B  max tensor(1)
E  min tensor(4)
tensor([ 1.3678,  1.1326,  1.4327, -1.0779, -1.0623, -1.4798, -1.0799,  1.0843,
         1.0784, -1.0534, -1.4296, -1.0375,  1.0665, -1.2190,  1.1062, -1.0608,
         1.2605, -1.3866,  1.3745,  1.0666]) tensor([ 1.3678,  1.1326,  1.4327, -1.0779, -1.0623, -1.4798, -1.0799,  1.0843,
         1.0784, -1.0534, -1.4296, -1.0375,  1.0665, -1.2190,  1.1062, -1.0608,
         1.2605, -1.3866,  1.3745,  1.0666]) tensor([ 1.3678,  1.1326,  1.4327, -1.0779, -1.0623, -1.4798, -1.0799,  1.0843,
         1.0784, -1.0534, -1.4296, -1.0375,  1.0665, -1.2190,  1.1062, -1.0608,
         1.2605, -1.3866,  1.3745,  1.0666])
C
B  max tensor(1)
F  min tensor(5)
tensor([ 1.3678,  1.1326,  1.4327, -1.0779, -1.0623, -1.4798, -1.0799,  1.0843,
         1.0784, -1.0534, -1.4296, -1.0375,  1.0665, -1.2190,  1.1062, -1.0608,
         1.2605, -1.3866,  1.3745,  1.0666]) tensor([ 1.3678,  1.1326,  1.4327, -1.0779, -1.0623, -1.4798, -1.0799,  1.0843,
         1.0784, -1.0534, -1

In [76]:
seq

'AGABCGBACGEDFGEFDGCBAGEFDGCBAGDFEGBCAGDFEGCBAGDFEGBACGCABGFFEDGABCGBCAGDEFGBACGFEDGFEDGDEFGCBAGFDEGABCGABCGEDFGEFDGBACGCABGEFDGEFDGEFDGFEDGEFDGFDEGDFEGAFEGDEFGEFDGCBAGACBGDFEGDFEGABCGCBAGDFEGDFEGBCAGEFDGDFEGEFDGBACGBACGFEDGEFDGDEFGEFDGCABGEFDDEFDFEGFEDGCABGBCAGFEDGEDFGEDFGEDFGACBGABCGBCAGCBAGBCAGEDFGEFDGACBGFDEGABCGABCGCBAGFEDGEDFGDEFGBACGEFDGFDEGEFDGDEFGEDFGBCAGEFDGCBBGBCAGBCAGBACGCBAGBACGFEDGBACGFDEGBACGABCGCBAGDFEGDEFGBCAGDFEGBCAGBACGFDEGFEDGABCGCABGBCAGEFDGDFEGBACGDFEGCBAGABCGDFEGFDEGBCAGABCGFEDGDEFGBACGCBAGABCGDFEGDFEGBCAGCABGBCAGFDEGDEFGBCAGABCGABCGABCGBCAGEDFGEFDGCABGCBAGDFEGEFDGDFEGBCAGEFDGBCAGDFEGDFEGDFEGBCAGDFEGDEFGEDFGACBGDFEGCABGACBGDFEGDFEGCABGACBGACBGEDFGCBAGFDEGBACGFEDGDEFGFEDGDFEGCABGDFEGABCGBACGDFEGBCAGBACGEFDGFDEGCABGEFDGACBGFEDGEDFGDFEGBACGEDFGBCAGCBAGABCGDEFGCBAGDFEGBACGABCGDFEGDFEGEDFGBCAGFEDGDEFGDEFGEFDGBCAGEFDGBACGCABGCABGDFEGFEDGDFEGEFDGFDEGDFEGABCGEFDGBACGFEDGDFEGABCGACBGBCAGFEDGFDEGDFEGDFEGFDEGACBGFEDGBACGEDFGCBAGEFDGEFDGEFDGEFDGBACGEDFGBACGEDFGCABGEDFGCBAGDF

In [23]:
x_sleep.reshape(1,1,-1)

tensor([[[-0.5654,  6.1418, -1.3794,  0.0995, -1.8208]]])

In [50]:
compute_geodesic(out1,out2)

ValueError: XA must be a 2-dimensional array.

In [51]:
out1.shape

torch.Size([20])

In [64]:
compute_geodesic(hidden_state_pos, hidden_state)

np.float64(0.6799006307013105)